In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"Libraries imported successfully!")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Libraries imported successfully!
Device: cuda
GPU: NVIDIA GeForce RTX 5090


In [2]:
ratings = pd.read_csv('../data/ml-25m/ratings.csv')
movies = pd.read_csv('../data/ml-25m/movies.csv')
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

split_date = '2015-01-01'
train = ratings[ratings['datetime'] < split_date].copy()
test = ratings[ratings['datetime'] >= split_date].copy()

min_rating = 0.5
max_rating = 5.0
train['rating'] = (train['rating'] - min_rating) / (max_rating - min_rating)
test['rating'] = (test['rating'] - min_rating) / (max_rating - min_rating)

print(f"Train: {len(train):,} ratings")
print(f"Test: {len(test):,} ratings")
print(f"Movies: {len(movies):,}")
print(f"Rating range: {train['rating'].min():.2f} - {train['rating'].max():.2f}")

Train: 17,436,354 ratings
Test: 7,563,741 ratings
Movies: 62,423
Rating range: 0.00 - 1.00


In [3]:
user_ids = train['userId'].unique()
movie_ids = train['movieId'].unique()

user_id_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

index_to_user_id = {idx: user_id for user_id, idx in user_id_to_index.items()}
index_to_movie_id = {idx: movie_id for movie_id, idx in movie_id_to_index.items()}

train['user_index'] = train['userId'].map(user_id_to_index)
train['movie_index'] = train['movieId'].map(movie_id_to_index)
test['user_index'] = test['userId'].map(user_id_to_index)
test['movie_index'] = test['movieId'].map(movie_id_to_index)

n_users = len(user_ids)
n_movies = len(movie_ids)

print(f"Number of users:  {n_users:,}")
print(f"Number of movies: {n_movies:,}")
print(f"Test cold start users: {test['user_index'].isna().sum():,}")

Number of users:  121,673
Number of movies: 22,316
Test cold start users: 6,836,326


In [4]:
class MovieRatingDataset(Dataset):
    def __init__(self, dataframe):
        valid = dataframe['user_index'].notna() & dataframe['movie_index'].notna()
        dataframe = dataframe[valid].reset_index(drop=True)
        
        self.users = torch.tensor(
            dataframe['user_index'].values, dtype=torch.long)
        self.movies = torch.tensor(
            dataframe['movie_index'].values, dtype=torch.long)
        self.ratings = torch.tensor(
            dataframe['rating'].values, dtype=torch.float32)
        
    def __len__(self):
        return len(self.ratings)
    
    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

train_dataset = MovieRatingDataset(train)
test_dataset = MovieRatingDataset(test)

train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(f"Train dataset: {len(train_dataset):,}")
print(f"Test dataset:  {len(test_dataset):,}")
print(f"Train batches: {len(train_loader):,}")
print(f"Batch size:    4,096")

Train dataset: 17,436,354
Test dataset:  546,990
Train batches: 4,257
Batch size:    4,096


In [5]:
class GMF(nn.Module):
    def __init__(self, n_users, n_movies, embedding_dim=32, dropout=0.3):
        super(GMF, self).__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.movie_embedding = nn.Embedding(n_movies, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.movie_embedding.weight, std=0.1)
        self.dropout = nn.Dropout(p=dropout)
        self.bn = nn.BatchNorm1d(embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, 1)
        
    def forward(self, user_indices, movie_indices):
        user_vec = self.user_embedding(user_indices)
        movie_vec = self.movie_embedding(movie_indices)
        interaction = user_vec * movie_vec
        interaction = self.dropout(self.bn(interaction))
        output = self.output_layer(interaction)
        return output.squeeze()

# Load saved weights
gmf_model = GMF(n_users, n_movies, embedding_dim=32, dropout=0.3).to(device)
gmf_model.load_state_dict(
    torch.load('../models/gmf_regularized_best.pth', map_location=device))
gmf_model.eval()

print(f"GMF Regularized model loaded")
print(f"Best RMSE: 0.8340")

GMF Regularized model loaded successfully!
Best RMSE: 0.8340


In [6]:
all_genres = set()
for genres in movies['genres'].str.split('|'):
    all_genres.update(genres)

# Remove no genres listed
all_genres.discard('(no genres listed)')
all_genres = sorted(list(all_genres))

print(f"Unique genres found: {len(all_genres)}")
print(f"Genres: {all_genres}")

# Create multi-hot encoding for each movie
def create_genre_vector(genres_str, all_genres):
    genres = genres_str.split('|')
    vector = np.zeros(len(all_genres), dtype=np.float32)
    for genre in genres:
        if genre in all_genres:
            idx = all_genres.index(genre)
            vector[idx] = 1.0
    return vector

# Apply to all movies
print("\nCreating genres")
movies['genre_vector'] = movies['genres'].apply(
    lambda x: create_genre_vector(x, all_genres))

# Create movie_id to genre vector mapping
movie_id_to_genre = dict(zip(movies['movieId'], movies['genre_vector']))

print(f"Genre vectors created for {len(movie_id_to_genre):,} movies")
print(f"Vector size: {len(all_genres)} dimensions")
print(f"\nExample - Toy Story genres:")
toy_story = movies[movies['title'].str.contains('Toy Story', na=False)].iloc[0]
print(f"  Title:  {toy_story['title']}")
print(f"  Genres: {toy_story['genres']}")
print(f"  Vector: {toy_story['genre_vector']}")

Unique genres found: 19
Genres: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Creating genre vectors...
Genre vectors created for 62,423 movies
Vector size: 19 dimensions

Example - Toy Story genres:
  Title:  Toy Story (1995)
  Genres: Adventure|Animation|Children|Comedy|Fantasy
  Vector: [0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [7]:
def get_genre_vector(genre_list, all_genres):
    """Convert list of genre strings to preference vector"""
    vector = np.zeros(len(all_genres), dtype=np.float32)
    for genre in genre_list:
        if genre in all_genres:
            idx = all_genres.index(genre)
            vector[idx] = 1.0
    return vector

def cold_start_recommendations(
        preferred_genres,
        all_genres,
        movie_id_to_genre,
        movies,
        train,
        top_k=10,
        min_ratings=50):
    
    #Create user preference vector
    user_vector = get_genre_vector(preferred_genres, all_genres)
    print(f"User preferences: {preferred_genres}")
    print(f"Preference vector: {user_vector}")
    
    #Similarity scores for all movies
    scores = {}
    for movie_id, genre_vector in movie_id_to_genre.items():
        # Dot product similarity
        similarity = np.dot(user_vector, genre_vector)
        if similarity > 0:
            scores[movie_id] = similarity
    
    print(f"\nMovies with matching genres: {len(scores):,}")
    
    #Average ratings and filter by popularity
    movie_stats = train.groupby('movieId')['rating'].agg(['mean', 'count'])
    movie_stats.columns = ['avg_rating', 'num_ratings']
    movie_stats['avg_rating'] = (movie_stats['avg_rating'] * 
                                  (max_rating - min_rating)) + min_rating
    
    #Combine similarity + popularity
    recommendations = []
    for movie_id, similarity in scores.items():
        if movie_id in movie_stats.index:
            stats = movie_stats.loc[movie_id]
            if stats['num_ratings'] >= min_ratings:
                # Combined score: genre match + rating quality
                combined_score = similarity * stats['avg_rating']
                recommendations.append({
                    'movieId': movie_id,
                    'similarity': similarity,
                    'avg_rating': round(stats['avg_rating'], 2),
                    'num_ratings': int(stats['num_ratings']),
                    'combined_score': combined_score
                })
    
    #Sort by combined score and return top K
    recommendations = sorted(recommendations, 
                             key=lambda x: x['combined_score'], 
                             reverse=True)[:top_k]
    
    #Add movie titles
    movie_titles = dict(zip(movies['movieId'], movies['title']))
    movie_genres_dict = dict(zip(movies['movieId'], movies['genres']))
    
    results = []
    for i, rec in enumerate(recommendations):
        results.append({
            'rank': i + 1,
            'title': movie_titles.get(rec['movieId'], 'Unknown'),
            'genres': movie_genres_dict.get(rec['movieId'], 'Unknown'),
            'avg_rating': rec['avg_rating'],
            'num_ratings': rec['num_ratings'],
            'genre_match': rec['similarity'],
            'combined_score': round(rec['combined_score'], 3)
        })
    
    return pd.DataFrame(results)

print(f"Uses genre similarity + popularity ranking")
print(f"Filters movies with < 50 ratings")

Uses genre similarity + popularity ranking
Filters movies with < 50 ratings


In [8]:
print("TEST 1: Sci-Fi + Thriller Fan")
print("=" * 60)
recs1 = cold_start_recommendations(
    preferred_genres=['Sci-Fi', 'Thriller'],
    all_genres=all_genres,
    movie_id_to_genre=movie_id_to_genre,
    movies=movies,
    train=train,
    top_k=10
)
print(recs1.to_string(index=False))

# Test 2 — Comedy and Romance fan
print("\n" + "=" * 60)
print("TEST 2: Comedy + Romance Fan")
print("=" * 60)
recs2 = cold_start_recommendations(
    preferred_genres=['Comedy', 'Romance'],
    all_genres=all_genres,
    movie_id_to_genre=movie_id_to_genre,
    movies=movies,
    train=train,
    top_k=10
)
print(recs2.to_string(index=False))

# Test 3 — Action and Adventure fan
print("\n" + "=" * 60)
print("TEST 3: Action + Adventure Fan")
print("=" * 60)
recs3 = cold_start_recommendations(
    preferred_genres=['Action', 'Adventure'],
    all_genres=all_genres,
    movie_id_to_genre=movie_id_to_genre,
    movies=movies,
    train=train,
    top_k=10
)
print(recs3.to_string(index=False))

TEST 1: Sci-Fi + Thriller Fan
User preferences: ['Sci-Fi', 'Thriller']
Preference vector: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0.]

Movies with matching genres: 11,349
 rank                                 title                                          genres  avg_rating  num_ratings  genre_match  combined_score
    1                    Matrix, The (1999)                          Action|Sci-Fi|Thriller        4.19        44249          2.0           8.381
    2                      Inception (2010) Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX        4.17        11236          2.0           8.333
    3                   Blade Runner (1982)                          Action|Sci-Fi|Thriller        4.14        26723          2.0           8.277
    4                  Prestige, The (2006)                   Drama|Mystery|Sci-Fi|Thriller        4.04         9576          2.0           8.075
    5                   Donnie Darko (2001)                   Drama|Mystery|Sci-Fi|T

 movieId                                                                          title                                     genres
    1200                                                                  Aliens (1986)             Action|Adventure|Horror|Sci-Fi
    1214                                                                   Alien (1979)                              Horror|Sci-Fi
    1320                                                 Alien³ (a.k.a. Alien 3) (1992)              Action|Horror|Sci-Fi|Thriller
    1690                                                     Alien: Resurrection (1997)                       Action|Horror|Sci-Fi
    1692                                                            Alien Escape (1995)                              Horror|Sci-Fi
    3701                                                            Alien Nation (1988)                Crime|Drama|Sci-Fi|Thriller
    4526                                               My Stepmother Is an Alien (1

In [14]:
def hybrid_recommendations(
        user_id,
        preferred_genres,
        all_genres,
        movie_id_to_genre,
        movies,
        train,
        gmf_model,
        top_k=10,
        min_ratings=50):
    
    #User is cold start or existing
    is_cold_start = user_id not in user_id_to_index
    
    if is_cold_start:
        print(f"User {user_id}: COLD START — using genre based recommendations")
        return cold_start_recommendations(
            preferred_genres, all_genres, movie_id_to_genre,
            movies, train, top_k, min_ratings)
    
    print(f"User {user_id}: EXISTING USER — using hybrid GMF + genre recommendations")
    
    #Get user index
    user_index = user_id_to_index[user_id]
    
    #Get movies user hasn't rated yet
    rated_movies = set(train[train['userId'] == user_id]['movieId'].values)
    
    #Get movie stats and filter by min_ratings
    movie_stats = train.groupby('movieId')['rating'].agg(['mean', 'count'])
    movie_stats['avg_rating'] = (movie_stats['mean'] * 
                                  (max_rating - min_rating)) + min_rating
    popular_movies = set(
        movie_stats[movie_stats['count'] >= min_ratings].index.tolist())
    
    candidate_movies = [
        mid for mid in movie_id_to_index.keys()
        if mid not in rated_movies and mid in popular_movies
    ]
    
    print(f"  Movies already rated:     {len(rated_movies):,}")
    print(f"  Candidate movies:         {len(candidate_movies):,}")
    
    #Create genre preference vector
    user_genre_vector = get_genre_vector(preferred_genres, all_genres)
    max_genre_match = len(preferred_genres)
    
    #Score all candidate movies with GMF
    gmf_model.eval()
    movie_indices = [movie_id_to_index[mid] for mid in candidate_movies]
    
    all_gmf_scores = []
    batch_size = 4096
    with torch.no_grad():
        for i in range(0, len(movie_indices), batch_size):
            batch_movie_indices = torch.tensor(
                movie_indices[i:i+batch_size], dtype=torch.long).to(device)
            batch_user_indices = torch.tensor(
                [user_index] * len(batch_movie_indices),
                dtype=torch.long).to(device)
            
            gmf_pred = gmf_model(batch_user_indices, batch_movie_indices)
            gmf_scores = (gmf_pred.cpu().numpy() * 
                         (max_rating - min_rating)) + min_rating
            all_gmf_scores.extend(gmf_scores)
    
    #Combine GMF + genre scores
    recommendations = []
    for idx, movie_id in enumerate(candidate_movies):
        genre_vector = movie_id_to_genre.get(movie_id, np.zeros(len(all_genres)))
        
        #Genre similarity
        genre_match = np.dot(user_genre_vector, genre_vector)
        
        #Normalize genre to 0-5 scale
        if max_genre_match > 0:
            normalized_genre = (genre_match / max_genre_match) * 5.0
        else:
            normalized_genre = 0.0
        
        gmf_score = all_gmf_scores[idx]
        
        if len(preferred_genres) == 0:
            hybrid_score = gmf_score
        else:
            hybrid_score = (gmf_score + normalized_genre) / 2
        
        recommendations.append({
            'movieId': movie_id,
            'gmf_score': round(float(gmf_score), 3),
            'genre_match': genre_match,
            'normalized_genre': round(normalized_genre, 3),
            'hybrid_score': round(hybrid_score, 3)
        })
    
    #Sort by hybrid score and return top K
    recommendations = sorted(recommendations,
                             key=lambda x: x['hybrid_score'],
                             reverse=True)[:top_k]
    
    #Add movie details
    movie_titles = dict(zip(movies['movieId'], movies['title']))
    movie_genres_dict = dict(zip(movies['movieId'], movies['genres']))
    
    results = []
    for i, rec in enumerate(recommendations):
        mid = rec['movieId']
        avg_rating = movie_stats.loc[mid, 'avg_rating'] if mid in movie_stats.index else 0
        num_ratings = movie_stats.loc[mid, 'count'] if mid in movie_stats.index else 0
        
        results.append({
            'rank': i + 1,
            'title': movie_titles.get(mid, 'Unknown'),
            'genres': movie_genres_dict.get(mid, 'Unknown'),
            'gmf_score': rec['gmf_score'],
            'genre_match': rec['genre_match'],
            'hybrid_score': rec['hybrid_score'],
            'avg_rating': round(avg_rating, 2),
            'num_ratings': int(num_ratings)
        })
    
    return pd.DataFrame(results)

print("  New users:      genre based cold start")
print("  Existing users: GMF predictions + genre similarity")
print("  Scoring:        GMF only (no genre) or average of both signals")
print("  Filter:         min 50 ratings required")

  New users:      genre based cold start
  Existing users: GMF predictions + genre similarity
  Scoring:        GMF only (no genre) or average of both signals
  Filter:         min 50 ratings required


In [ ]:
existing_user_id = train['userId'].iloc[0]
print("=" * 60)
print(f"TEST 1: EXISTING USER (userId: {existing_user_id})")
print("=" * 60)
recs1 = hybrid_recommendations(
    user_id=existing_user_id,
    preferred_genres=['Sci-Fi', 'Thriller'],
    all_genres=all_genres,
    movie_id_to_genre=movie_id_to_genre,
    movies=movies,
    train=train,
    gmf_model=gmf_model,
    top_k=10
)
recs1['num_ratings'] = recs1['num_ratings'].apply(lambda x: f"{x:,}")
print(recs1.to_string(index=False))

# Test 2 — Cold start user who likes Action and Adventure
fake_new_user_id = 999999999
print("\n" + "=" * 60)
print(f"TEST 2: NEW USER (userId: {fake_new_user_id})")
print("=" * 60)
recs2 = hybrid_recommendations(
    user_id=fake_new_user_id,
    preferred_genres=['Action', 'Adventure'],
    all_genres=all_genres,
    movie_id_to_genre=movie_id_to_genre,
    movies=movies,
    train=train,
    gmf_model=gmf_model,
    top_k=10
)
print(recs2.to_string(index=False))

# Test 3 — Existing user with no genre preference
print("\n" + "=" * 60)
print(f"TEST 3: EXISTING USER - NO GENRE PREFERENCE (userId: {existing_user_id})")
print("=" * 60)
recs3 = hybrid_recommendations(
    user_id=existing_user_id,
    preferred_genres=[],
    all_genres=all_genres,
    movie_id_to_genre=movie_id_to_genre,
    movies=movies,
    train=train,
    gmf_model=gmf_model,
    top_k=10
)
print(recs3.to_string(index=False))

## Notebook 9 Summary: Content-Based Features & Hybrid Recommender

| User Type | Approach | Signal |
|-----------|----------|--------|
| Existing user + genre preference | Hybrid GMF + Genre | Average of both signals |
| Existing user + no preference | Pure GMF | Personalized rating prediction |
| New user (cold start) | Genre + Popularity | Multi-hot encoding + avg rating |

### Genre Feature Engineering
- Extracted 19 unique genres from `movies.csv`
- Created multi-hot encoding — 19-dimensional binary vector per movie
- Each dimension = 1 if movie has that genre, 0 if not
- Example — Toy Story: `[0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0]`
  (Adventure, Animation, Children, Comedy, Fantasy)

### Cold Start Solution
New user selects preferred genres →
Genre similarity via dot product →
Filtered by popularity (min 50 ratings) →
Ranked by genre_match × avg_rating
```

**Cold start results for Action + Adventure fan:**
| Rank | Title | Avg Rating |
|------|-------|------------|
| 1 | Seven Samurai (1954) | 4.28 |
| 2 | City of God (2002) | 4.25 |
| 3 | North by Northwest (1959) | 4.24 |
| 4 | Raiders of the Lost Ark (1981) | 4.23 |

### Hybrid Scoring
For existing users with genre preferences:
```
Genre similarity → normalized to 0-5 scale
GMF prediction  → already on 0-5 scale
Hybrid score    → (GMF + normalized_genre) / 2
```
Equal weighting treats both signals as equally trustworthy.
If no genre preference provided → pure GMF score used.

### Key Findings
- Multi-hot encoding successfully captures multi-genre movies
- Dot product naturally rewards movies matching ALL requested genres
- min_ratings=50 filter prevents obscure films dominating results
- Hybrid scoring produces intuitive, explainable recommendations

### Known Limitations
1. Genre tags are inconsistent across MovieLens
   - Alien (1979) tagged as Horror|Sci-Fi not Thriller
   - Misses obvious Sci-Fi Thriller recommendations
2. Cold start still favors older critically acclaimed films
   - High avg ratings skew toward classic cinema
   - New users may not recognize recommendations
3. Equal weighting (50/50 GMF vs genre) not optimized
   - Could be tuned as a hyperparameter
   - More rating history → trust GMF more
   - Less history → trust genre more

### System Architecture
```
Input: user_id + preferred_genres
         
Is user in training set?
    YES → Hybrid (GMF + Genre)
    NO  → Cold Start (Genre + Popularity)
         
Output: Top 10 ranked recommendations
```